# Two readings of one document

A document has text and it has layout, and they are not the same information.
Extracting prose and losing the fact that four of those numbers were a table is
the standard failure of document pipelines, and it happens because the pipeline
was written as *one* pass.

Text and layout are two independent readings of the same bytes that meet when
fields are located. That is a diamond, and the model draws it as one.

In [1]:
# Installed from the repository, not from PyPI: this notebook uses `templates`
# and `viz`, which no published release contains yet. A notebook that installs
# something older than the API it calls fails at cell one, which is a confusing
# way to introduce a library about checking things before they run.
try:
    import browsergraph  # noqa: F401
except ImportError:  # pragma: no cover
    %pip install -q "browsergraph @ git+https://github.com/aidonerightcorp/browsergraph.git"

import browsergraph as bg
from browsergraph import templates as T, viz
from browsergraph.compile import CompileError, compile_route
from browsergraph.manifest import NodeManifest, ParameterSpec, PortSpec
from browsergraph.workbench import NodeCandidate

print("browsergraph", bg.__version__)

browsergraph 0.3.0


In [2]:
# From the library, not redefined here. A notebook that teaches helpers
# browsergraph does not have is a notebook nobody can build on.
from browsergraph.quick import chain, fanin, fanout, link, node, problems, step

In [3]:
template = T.get("document.extraction")
skeleton = template.skeleton()
print("layers:", skeleton.layers())
print("joins at:", [s.id for s in skeleton.leaf_stages if len(s.inputs) > 1])

layers: [['acquire'], ['detect'], ['text', 'layout'], ['fields'], ['normalise'], ['verify']]
joins at: ['fields']


In [4]:
nodes = [
    node("doc.acquire.file",  "io.read",       [], [("out", "Bytes")]),
    node("doc.acquire.blob",  "io.read",       [], [("out", "Bytes")],
         permissions=("storage.read",)),

    node("doc.detect.magic",  "doc.detect",    [("in", "Bytes")], [("out", "Document")]),

    node("doc.text.native",   "doc.text",      [("in", "Document")], [("out", "Text")]),
    node("doc.text.ocr",      "doc.text",      [("in", "Document")], [("out", "Text")],
         deterministic=False,
         facets={"purpose.not_for": ["born-digital PDFs"],
                 "cost.latency_ms": 2400.0}),

    node("doc.layout.rules",  "doc.layout",    [("in", "Document")], [("out", "Layout")]),
    node("doc.layout.model",  "doc.layout",    [("in", "Document")], [("out", "Layout")],
         deterministic=False),

    node("doc.fields.join",   "doc.fields",
         [("text", "Text"), ("layout", "Layout")], [("out", "Fields")]),
    node("doc.fields.llm",    "doc.fields",
         [("text", "Text"), ("layout", "Layout")], [("out", "Fields")],
         deterministic=False),

    node("doc.norm.strict",   "doc.normalise", [("in", "Fields")], [("out", "Record")]),
    node("doc.verify.totals", "doc.verify",    [("in", "Record")], [("out", "Record")]),
]

filling = {
    "acquire":   ["doc.acquire.file", "doc.acquire.blob"],
    "detect":    ["doc.detect.magic"],
    "text":      ["doc.text.native", "doc.text.ocr"],
    "layout":    ["doc.layout.rules", "doc.layout.model"],
    "fields":    ["doc.fields.join", "doc.fields.llm"],
    "normalise": ["doc.norm.strict"],
    "verify":    ["doc.verify.totals"],
}

bench = T.get("document.extraction").instantiate(filling)
bench = bench.__class__(**{**bench.__dict__, "nodes": tuple(nodes)})
print("routes:", bench.route_count())

routes: 16


In [5]:
viz.dag(bench)

Figure(svg='<svg viewBox="0 0 1170 308" width="1170" height="308" style="max-width:none" role="img"><defs><marker id="bg12209318-arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-end"><path d="M0,0 L10,5 L0,10 z" fill="#8a93a0"/></marker></defs><text x="153.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 0</text><g><rect x="60" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="69" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Acquire</text><text x="69" y="149.0" font-size="9.5" fill="#68737f">2 candidates</text></g><text x="363.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 1</text><g><rect x="270" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="279" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Detect format</text><text x="279" y="149.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="573.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 2 · 2 parallel</text><g><rect x="480" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="489" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Extract text</text><text x="489" y="108.0" font-size="9.5" fill="#68737f">2 candidates</text></g><g><rect x="480" y="156.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="489" y="175.0" font-size="11.5" font-weight="700" fill="#22303f">Extract layout</text><text x="489" y="190.0" font-size="9.5" fill="#68737f">2 candidates</text></g><text x="783.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 3</text><g><rect x="690" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="699" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Locate fields</text><text x="699" y="149.0" font-size="9.5" fill="#68737f">2 candidates</text></g><text x="993.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 4</text><g><rect x="900" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="909" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Normalise</text><text x="909" y="149.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="1203.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 5</text><g><rect x="1110" y="115.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="1119" y="134.0" font-size="11.5" font-weight="700" fill="#22303f">Verify</text><text x="1119" y="149.0" font-size="9.5" fill="#68737f">1 candidate</text></g><path d="M246,141.0 C258.0,141.0 258.0,141.0 270,141.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg12209318-arrow)"/><path d="M456,141.0 C468.0,141.0 468.0,100.0 480,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg12209318-arrow)"/><path d="M456,141.0 C468.0,141.0 468.0,182.0 480,182.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg12209318-arrow)"/><path d="M666,100.0 C678.0,100.0 678.0,141.0 690,141.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg12209318-arrow)"/><text x="678.0" y="115.5" text-anchor="middle" font-size="9" fill="#68737f">text</text><path d="M666,182.0 C678.0,182.0 678.0,141.0 690,141.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg12209318-arrow)"/><text x="678.0" y="156.5" text-anchor="middle" font-size="9" fill="#68737f">layout</text><path d="M876,141.0 C888.0,141.0 888.0,141.0 900,141.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg12209318-arrow)"/><path d="M1086,141.0 C1098.0,141.0 1098.0,141.0 1110,141.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" 

## A route that is fully deterministic, and one that is not

Determinism is not a property you assert about a pipeline — it is a property of
the nodes you chose, and it changes as you swap them.

In [6]:
safe = {"acquire": "doc.acquire.file", "detect": "doc.detect.magic",
        "text": "doc.text.native", "layout": "doc.layout.rules",
        "fields": "doc.fields.join", "normalise": "doc.norm.strict",
        "verify": "doc.verify.totals"}

scanned = {**safe, "text": "doc.text.ocr", "fields": "doc.fields.llm"}

for label, route in (("born-digital", safe), ("scanned", scanned)):
    plan = compile_route(bench, route)
    print(f"{label:<14} deterministic={plan.deterministic!s:<5} "
          f"permissions={plan.permissions or '—'}  {plan.digest[:20]}…")

born-digital   deterministic=True  permissions=—  plan:822ad68b072bc26…
scanned        deterministic=False permissions=—  plan:04986ebb16c39fc…


## The join is checked on the ports, not on hope

Cross-wire the two readings and the compiler names the edge and the mismatch.

In [7]:
from browsergraph.workbench import Edge

crossed = bench.__class__(**{**bench.__dict__, "edges": (
    Edge("acquire", "detect"), Edge("detect", "text"), Edge("detect", "layout"),
    Edge("text", "fields", to_port="layout"),      # swapped
    Edge("layout", "fields", to_port="text"),      # swapped
    Edge("fields", "normalise"), Edge("normalise", "verify"))})

try:
    compile_route(crossed, safe)
except CompileError as exc:
    for problem in exc.problems:
        print("refused:", problem)

refused: edge text -> fields.layout: 'Text' is not a 'Layout' — declare the subtype relation if it holds, or insert an adapter node
refused: edge layout -> fields.text: 'Layout' is not a 'Text' — declare the subtype relation if it holds, or insert an adapter node


In [8]:
viz.route_space(bench, route=safe, alternative=scanned)

Figure(svg='<svg viewBox="0 0 1180 230" width="1180" height="230" style="max-width:none" role="img"><style>.bg43511980-v{cursor:pointer}.bg43511980-v:hover rect{stroke:#c0392b;stroke-width:2}</style><polyline points="46,96 170,96 206,96 330,96 366,96 490,96 526,96 650,96 686,96 810,96 846,96 970,96 1006,96 1130,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,126 170,126 206,96 330,96 366,96 490,96 526,96 650,96 686,96 810,96 846,96 970,96 1006,96 1130,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,96 170,96 206,96 330,96 366,126 490,126 526,96 650,96 686,96 810,96 846,96 970,96 1006,96 1130,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,126 170,126 206,96 330,96 366,126 490,126 526,96 650,96 686,96 810,96 846,96 970,96 1006,96 1130,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,96 170,96 206,96 330,96 366,96 490,96 526,126 650,126 686,96 810,96 846,96 970,96 1006,96 1130,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,126 170,126 206,96 330,96 366,96 490,96 526,126 650,126 686,96 810,96 846,96 970,96 1006,96 1130,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,96 170,96 206,96 330,96 366,126 490,126 526,126 650,126 686,96 810,96 846,96 970,96 1006,96 1130,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,126 170,126 206,96 330,96 366,126 490,126 526,126 650,126 686,96 810,96 846,96 970,96 1006,96 1130,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,96 170,96 206,96 330,96 366,96 490,96 526,96 650,96 686,126 810,126 846,96 970,96 1006,96 1130,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,126 170,126 206,96 330,96 366,96 490,96 526,96 650,96 686,126 810,126 846,96 970,96 1006,96 1130,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,96 170,96 206,96 330,96 366,126 490,126 526,96 650,96 686,126 810,126 846,96 970,96 1006,96 1130,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,126 170,126 206,96 330,96 366,126 490,126 526,96 650,96 686,126 810,126 846,96 970,96 1006,96 1130,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,96 170,96 206,96 330,96 366,96 490,96 526,126 650,126 686,126 810,126 846,96 970,96 1006,96 1130,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,126 170,126 206,96 330,96 366,96 490,96 526,126 650,126 686,126 810,126 846,96 970,96 1006,96 1130,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,96 170,96 206,96 330,96 366,126 490,126 526,126 650,126 686,126 810,126 846,96 970,96 1006,96 1130,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,126 170,126 206,96 330,96 366,126 490,126 526,126 650,126 686,126 810,126 846,96 970,96 1006,96 1130,96" fill="none" stroke="#2d6cb5" stroke-width="1" opacity=".07"/><polyline points="46,96 170,96 206,96 330,96 366,96 490,96 526,96 650,96 686,96 810,96 846,96 970,96 1006,96 1130,96" fill="none" stroke="#c0392b" stroke-width="2.6" opacity=".95"/><polyline points="46,96 170,96 206,96 330,96 366,126 490,126 526,96 650,96 686,126 810,126 846,96 970,96 1006,96 1130,96" fill="none" stroke="#c98a2b" stroke-width="2.6" opacity=".95" stroke-dasharray="7 4"/><line x1="108" y1="62" x2="108" y2="174" stroke="#dfe5ec" stroke-width="1"/><text x="108" y="46" text-anchor="middle" font-size="12" font-weight="700" fill="#22303f">Acquire</text><text x="108" y="58" text-anchor="middle" font-size="9" fill="#68737f">2 options</text><g class="bg43511980-v" data-stage="acquire" data-cid="doc.acquire.file"><rect x="46" y="86" width="124" height="21" rx="5" fill="#fdeceb" stroke="#c0392b" stroke-width="1"/><text x="108" y="101" text-anchor="middle" 

## Evidence is per step, not per run

A route that failed tells you one bit: something was wrong. Per-step outcomes
tell you *where*, and that difference is what makes learning across runs
practical rather than theoretical.

In [9]:
viz.evidence({
    "acquire":   0.9,
    "detect":    0.4,
    "text":      1.8,
    "layout":   -2.1,   # the table came back as prose
    "fields":   -0.6,
    "normalise": 0.2,
    "verify":    1.1,
}, title="document extraction — bits per step")

Figure(svg='<svg viewBox="0 0 940 308" width="940" height="308" style="max-width:none" role="img"><line x1="525.0" y1="44" x2="525.0" y2="278" stroke="#dfe5ec" stroke-width="1"/><text x="525.0" y="294" text-anchor="middle" font-size="9.5" fill="#68737f">0 bits</text><text x="184" y="73" text-anchor="end" font-size="11" fill="#22303f">acquire</text><rect x="525.0" y="60" width="139.3" height="18" rx="3" fill="#1f8a4c" opacity=".72"/><text x="672.3" y="73" font-size="10" text-anchor="start" fill="#68737f">+0.90</text><text x="184" y="103" text-anchor="end" font-size="11" fill="#22303f">detect</text><rect x="525.0" y="90" width="61.9" height="18" rx="3" fill="#1f8a4c" opacity=".72"/><text x="594.9" y="103" font-size="10" text-anchor="start" fill="#68737f">+0.40</text><text x="184" y="133" text-anchor="end" font-size="11" fill="#22303f">text</text><rect x="525.0" y="120" width="278.6" height="18" rx="3" fill="#1f8a4c" opacity=".72"/><text x="811.6" y="133" font-size="10" text-anchor="start" fill="#68737f">+1.80</text><text x="184" y="163" text-anchor="end" font-size="11" fill="#22303f">layout</text><rect x="200.0" y="150" width="325.0" height="18" rx="3" fill="#c0392b" opacity=".72"/><text x="192.0" y="163" font-size="10" text-anchor="end" fill="#68737f">-2.10</text><text x="184" y="193" text-anchor="end" font-size="11" fill="#22303f">fields</text><rect x="432.1" y="180" width="92.9" height="18" rx="3" fill="#c0392b" opacity=".72"/><text x="424.1" y="193" font-size="10" text-anchor="end" fill="#68737f">-0.60</text><text x="184" y="223" text-anchor="end" font-size="11" fill="#22303f">normalise</text><rect x="525.0" y="210" width="31.0" height="18" rx="3" fill="#1f8a4c" opacity=".72"/><text x="564.0" y="223" font-size="10" text-anchor="start" fill="#68737f">+0.20</text><text x="184" y="253" text-anchor="end" font-size="11" fill="#22303f">verify</text><rect x="525.0" y="240" width="170.2" height="18" rx="3" fill="#1f8a4c" opacity=".72"/><text x="703.2" y="253" font-size="10" text-anchor="start" fill="#68737f">+1.10</text></svg>', title='document extraction — bits per step', note='Positive: this step supported the route. Negative: it argued against it.', width=940, height=308)

Layout is the problem and the picture says so. A pass/fail signal on the whole
route would have said "this route is bad" and left seven suspects.